---
## 1. Normalization in Transformers (After Multi-Head Attention)

In Transformer architectures, normalization plays a crucial role in stabilizing training and ensuring smooth gradient flow. 

---

### 1.1. Multi-Head Attention (MHA) Recap

Given an input $X \in \mathbb{R}^{\text{seq\_len} \times d_\text{model}}$, MHA computes:

$$
\text{Attention}(Q, K, V) = \text{softmax}\Big(\frac{Q K^T}{\sqrt{d_k}}\Big) V
$$

- $Q, K, V$ are linear projections of $X$.
- Output of MHA has the same shape as input: $\mathbb{R}^{\text{seq\_len} \times d_\text{model}}$.

---

### 1.2. Why Normalization is Needed After MHA

- MHA output can have **large variations in magnitude** depending on input sequence.
- Without normalization, deeper Transformer layers may experience:
  - Gradient explosion or vanishing
  - Unstable training
  - Difficulty learning residual connections

- Normalization ensures:
  - Each feature across the embedding dimension is **centered and scaled**
  - Smooth gradients for residual connections

---

### 1.3. LayerNorm in Transformers

In most modern Transformers (e.g., **GPT, BERT**):

$$
\text{Output} = \text{LayerNorm}(X + \text{MHA}(X))
$$

- **Residual connection**: $X + \text{MHA}(X)$
- **LayerNorm** normalizes **per token across features**:
  - For a token embedding $x \in \mathbb{R}^{d_\text{model}}$:
    $$
    \hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \odot \hat{x} + \beta
    $$
- Ensures that each token’s embedding is **well-scaled** before going to the feedforward network (FFN).

---

### 1.4. Why Not BatchNorm in Transformers?

- BatchNorm normalizes **across the batch**, but in Transformers:
  - Batch size may vary (small batches or even batch=1 during inference)
  - Sequences have **variable lengths**
  - Batch statistics would be inconsistent
- LayerNorm works **independently for each token**, making it ideal.

---


## 2. Batch Normalization vs Layer Normalization

Normalization techniques are used to **stabilize and accelerate training** in neural networks by normalizing inputs of layers.


### Comparision  Summary

| Feature / Property          | Batch Normalization (BatchNorm) | Layer Normalization (LayerNorm) |
|-----------------------------|--------------------------------|--------------------------------|
| **Normalization axis**      | Across the batch (per feature) | Across features (per sample/token) |
| **Formula**                 | $$\hat{x}^{(k)} = \frac{x^{(k)} - \mu_k}{\sqrt{\sigma_k^2 + \epsilon}}, \quad y^{(k)} = \gamma_k \hat{x}^{(k)} + \beta_k$$<br>where:<br>$$\mu_k = \frac{1}{N}\sum_{i=1}^N x_i^{(k)}, \quad \sigma_k^2 = \frac{1}{N}\sum_{i=1}^N (x_i^{(k)} - \mu_k)^2$$ | $$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \odot \hat{x} + \beta$$<br>where:<br>$$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \quad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2$$ |
| **Learnable parameters**    | γ (scale), β (shift) per feature | γ (scale), β (shift) per feature |
| **Batch size sensitivity**  | Sensitive, needs reasonably large batch | Independent of batch size |
| **Typical use cases**       | CNNs, feedforward networks       | RNNs, Transformers, sequential models |
| **Input shape requirement** | Fixed batch size                 | Works with variable batch size |
| **Residual connections**    | Can be used, but less common in sequence models | Often used after MHA and FFN layers in Transformers |
| **Effect**                  | Stabilizes learning across batches | Stabilizes learning per sample/token |
| **Computation**             | Requires batch statistics       | Requires statistics per sample |



---

### Example Calculation

$$
X = 
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9 \\
10 & 11 & 12
\end{bmatrix}
$$

Shape: **(batch = 4, features = 3)**



In [1]:
import torch
import torch.nn as nn

# Batch of 4 samples, each with 3 features
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10.0, 11.0, 12.0]
])

print("Input:\n", x)

Input:
 tensor([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [10., 11., 12.]])


---

## 🔹 Batch Normalization


$$
\hat{x}^{(k)} = \frac{x^{(k)} - \mu_k}{\sqrt{\sigma_k^2 + \epsilon}}, \quad
y^{(k)} = \gamma_k \hat{x}^{(k)} + \beta_k
$$

where $\mu_k$ and $\sigma_k^2$ are computed **over the batch** for feature $k$.

- **Key Points**:
  - Normalizes **each feature across the batch**.
  - $\gamma$ and $\beta$ are learnable parameters per feature.
  - Works best when batch size is sufficiently large.

- **When to use**:
  - CNNs (Convolutional Neural Networks)
  - Feedforward networks with large batch sizes
  - When you want **training acceleration and regularization**
---

### Example

$$
X = 
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9 \\
10 & 11 & 12
\end{bmatrix}
$$

**Step 1. Compute mean & variance per feature (column-wise).**

- Feature 1:  $[ 1, 4, 7, 10]$

  $\mu_1 = \frac{1+4+7+10}{4} = 5.5$  
  $\sigma_1^2 = \frac{(1-5.5)^2 + (4-5.5)^2 + (7-5.5)^2 + (10-5.5)^2}{4} = 8.25$

- Feature 2:  
  $\mu_2 = 6.5,\;\; \sigma_2^2 = 8.25$

- Feature 3:  
  $\mu_3 = 7.5,\;\; \sigma_3^2 = 8.25$

---

**Step 2. Normalize each value:**

$$
\hat{x}_{ij} = \frac{x_{ij} - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}}
$$

$$
\hat{x}_{11} = \frac{1 - 5.5}{\sqrt{8.25}} \approx -1.56
$$

---

**Final BatchNorm Output:**

$$
\hat{X}_{BN} \approx
\begin{bmatrix}
-1.343 & -1.343 & -1.343 \\
-0.447 & -0.447 & -0.447 \\
0.447 & 0.447 & 0.447 \\
1.343 & 1.343 & 1.343
\end{bmatrix}
$$


👉 Each **column** has mean $0$ and variance $1$.

---

``batch_norm = nn.BatchNorm1d(num_features=3, affine=False)``
- affine=True → there ARE learnable parameters (γ and β)

- affine=False → there are NO learnable parameters

---

In [17]:
# BatchNorm1d (for features=3)
batch_norm = nn.BatchNorm1d(num_features=3, affine=True)
out_bn = batch_norm(x)

print("\nBatchNorm output With Affine :\n", out_bn)



BatchNorm output With Affine :
 tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]], grad_fn=<NativeBatchNormBackward0>)


In [16]:
# BatchNorm1d (for features=3)
batch_norm = nn.BatchNorm1d(num_features=3, affine=False)  # no gamma/beta
out_bn = batch_norm(x)

print("\nBatchNorm output:\n", out_bn)
print("\nColumn means (should be ~0):", out_bn.mean(dim=0))
print("Column variances (should be ~1):", out_bn.var(dim=0, unbiased=False))



BatchNorm output:
 tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])

Column means (should be ~0): tensor([-2.9802e-08,  5.9605e-08,  1.1921e-07])
Column variances (should be ~1): tensor([1.0000, 1.0000, 1.0000])


---

## 🔹 Layer Normalization

For a single input sample $x \in \mathbb{R}^d$ (a vector of features), **Layer Normalization** normalizes across **features**:

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$

where the mean and variance are computed per sample:

$$
\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \quad
\sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2
$$

After normalization, a learnable **scale** ($\gamma$) and **shift** ($\beta$) are applied:


$$
y = \gamma \odot \hat{x} + \beta
$$

- $\gamma$ → scale parameter (learnable)  
- $\beta$ → shift parameter (learnable)  
- $\epsilon$ → small constant to avoid division by zero

---

- **Key Points**:
  - Normalizes **features within a single sample**.
  - $\gamma$ and $\beta$ are learnable per feature.
  - Independent of batch size (works well even for batch size = 1).

- **When to use**:
  - RNNs, Transformers (sequential models)
  - Small batch sizes
  - Situations where batch statistics are unreliable

---

### Example

$$
X = 
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9 \\
10 & 11 & 12
\end{bmatrix}
$$

**Step 1. Compute mean & variance per sample (row-wise).**

- Sample 1: $[1,2,3]$  
  $\mu = 2.0,\;\; \sigma^2 = 0.67$

- Sample 2: $[4,5,6]$  
  $\mu = 5.0,\;\; \sigma^2 = 0.67$

- Sample 3: $[7,8,9]$  
  $\mu = 8.0,\;\; \sigma^2 = 0.67$

- Sample 4: $[10,11,12]$  
  $\mu = 11.0,\;\; \sigma^2 = 0.67$

---

**Step 2. Normalize each value:**

$$
\hat{x}_{ij} = \frac{x_{ij} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}
$$

Example:  

$$
\hat{x}_{11} = \frac{1 - 2}{\sqrt{0.67}} \approx -1.22
$$

---

**Final LayerNorm Output:**

$$
\hat{X}_{LN} =
\begin{bmatrix}
-1.22 & 0.00 & +1.22 \\
-1.22 & 0.00 & +1.22 \\
-1.22 & 0.00 & +1.22 \\
-1.22 & 0.00 & +1.22
\end{bmatrix}
$$

👉 Each **row** has mean $0$ and variance $1$.

---


``layer_norm = nn.LayerNorm(normalized_shape=3, elementwise_affine=False)``
- elementwise_affine=True → there ARE learnable parameters (γ and β)

- elementwise_affine=False → there are NO learnable parameters

---

In [18]:
# LayerNorm (normalize across each row)
layer_norm = nn.LayerNorm(normalized_shape=3, elementwise_affine=True)
out_ln = layer_norm(x)

print("\nLayerNorm output with affin (scale and shift ):\n", out_ln)


LayerNorm output with affin (scale and shift ):
 tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]], grad_fn=<NativeLayerNormBackward0>)


In [19]:
# LayerNorm (normalize across each row)
layer_norm = nn.LayerNorm(normalized_shape=3, elementwise_affine=False)
out_ln = layer_norm(x)

print("\nLayerNorm output:\n", out_ln)
print("\nRow means (should be ~0):", out_ln.mean(dim=1))
print("Row variances (should be ~1):", out_ln.var(dim=1, unbiased=False))



LayerNorm output:
 tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])

Row means (should be ~0): tensor([0., 0., 0., 0.])
Row variances (should be ~1): tensor([1.0000, 1.0000, 1.0000, 1.0000])
